# 01 — CICIDS2017 Preprocessing
## 1. Introduction
CICIDS2017 gồm 8 CSV. Notebook tái chạy pipeline hiện tại để audit, làm sạch,
chia theo ngày và lưu dữ liệu chuẩn bị cho bước training sau.
Isolation Forest sẽ được xử lý trong notebook sau.

**Restart Kernel → Run All** từ đầu. Không cần biến từ phiên chạy trước.
Notebook gọi nguyên `src.run_preprocessing.run()` tại phần 4 rồi trình bày báo cáo vừa sinh
theo các phần bên dưới. Các phần 5–18 là trình bày kết quả của lần chạy đó,
không thực hiện lại thuật toán hay đổi thứ tự trong `src/`.

Trong pipeline thực tế, feature audit/loại cột dựa trên Train diễn ra **trước row cleanup**.
Phần 15 trình bày lại kết quả này; không chọn feature lần thứ hai.

## 2. Imports
Chọn kernel Python đã cài `requirements.txt`. Tìm project theo cấu trúc thư mục, không dùng đường dẫn cá nhân.

In [1]:
import sys
from pathlib import Path
import json
import logging

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'src' / 'run_preprocessing.py').is_file()
                     and (p / 'data' / 'raw').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Mở notebook từ project hoặc thư mục notebooks của project.')
sys.path.insert(0, str(PROJECT_ROOT))
# Các dependency cài cục bộ là tùy chọn, không phụ thuộc đường dẫn máy cá nhân.
for folder in ('.notebook-deps', '.runtime-deps'):
    local_packages = PROJECT_ROOT / folder
    if local_packages.is_dir():
        sys.path.insert(0, str(local_packages))

import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from IPython.display import display, Markdown
from src import load_data, data_audit, clean_data, feature_audit, split_data, validation
from src.config import SOURCES, EXCLUDED, IDENTIFIERS
from src.run_preprocessing import run

def read_report(filename):
    """Đọc báo cáo của lần chạy hiện tại để trình bày."""
    return json.loads((REPORT_DIR / filename).read_text(encoding='utf-8'))

def statistics_table(statistics):
    """Định dạng thống kê có sẵn; không thay đổi dữ liệu."""
    return pd.DataFrame([{'Split': name, 'Rows': item['total'],
                         'BENIGN': item['benign'], 'ATTACK': item['attack']}
                        for name, item in statistics.items()])

## 3. Configuration
Mapping và quy tắc được import từ `src.config`; giới hạn NaN/Inf thật giữ mặc định 1%.

In [2]:
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
OUTPUT_DIR = PROJECT_ROOT / 'data'
REPORT_DIR = PROJECT_ROOT / 'reports'
SOURCE_DAY_MAPPING = dict(SOURCES)
MAX_NON_FINITE_FRACTION = 0.01
EXPECTED_COUNTS = {'train': 422352, 'calibration': 105730,
                   'selection': 1582723, 'test': 700153}
assert all((RAW_DATA_DIR / name).is_file() for name in SOURCE_DAY_MAPPING)
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
display(pd.DataFrame(SOURCE_DAY_MAPPING.items(), columns=['File', 'source_day']))

,File,source_day
0,Monday-WorkingHours.pcap_ISCX.csv,Monday
1,Tuesday-WorkingHours.pcap_ISCX.csv,Tuesday
2,Wednesday-workingHours.pcap_ISCX.csv,Wednesday
3,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,Thursday-WebAttack
4,Thursday-WorkingHours-Afternoon-Infilteration....,Thursday-Infiltration
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,Friday-Bot
6,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Friday-DDoS
7,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Friday-PortScan


## 4. Load Raw Dataset
Cell này chạy **toàn bộ pipeline gốc một lần**, từ 8 raw CSV tới Parquet và báo cáo.
`run()` gọi `load_data.load_all_datasets()` và các module hiện tại, không đọc cache thay raw.
Nếu có lỗi, Run All dừng; không dùng báo cáo cũ để coi là thành công.
Các cell sau chỉ hiển thị kết quả và thực hiện kiểm tra reproduction.

In [3]:
run(RAW_DATA_DIR, OUTPUT_DIR, REPORT_DIR, MAX_NON_FINITE_FRACTION)
overview = read_report('dataset_overview.json')
cleaning = read_report('cleaning_report.json')
display(pd.DataFrame(overview)[['file', 'physical_rows', 'comma_only_rows',
                               'structured_rows', 'benign', 'attack']]
        .rename(columns={'benign': 'BENIGN', 'attack': 'ATTACK'}))

2026-10-13 16:47:28,915 INFO Audit Monday-WorkingHours.pcap_ISCX.csv: physical rows=529918 malformed=0 structured rows=529918 memory=407014993


2026-10-13 16:47:36,967 INFO Audit Tuesday-WorkingHours.pcap_ISCX.csv: physical rows=445909 malformed=0 structured rows=445909 memory=339862969


2026-10-13 16:47:50,300 INFO Audit Wednesday-workingHours.pcap_ISCX.csv: physical rows=692703 malformed=0 structured rows=692703 memory=527496238


2026-10-13 16:47:54,348 INFO Audit Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: physical rows=458968 malformed=288602 structured rows=170366 memory=130021990


2026-10-13 16:47:58,842 INFO Audit Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: physical rows=288602 malformed=0 structured rows=288602 memory=219889571


...

2026-10-13 16:50:43,574 INFO Split selection: rows before=1582723 after=1582723 memory bytes=896448360


2026-10-13 16:50:43,576 INFO Split test: rows before=700153 after=700153 memory bytes=425155980


2026-10-13 16:51:15,777 INFO Parquet saved: rows before=2810958 after=2810958 memory bytes=1529838318


CICIDS2017 PREPROCESSING SUMMARY
Raw rows: 3119345
Cleaned rows: 2810958
Train: 422352
Calibration: 105730
Selection: 1582723
Test: 700153
Train attacks: 0
Calibration attacks: 0
NaN remaining: 0
Inf remaining: 0
Non-numeric model features: []
Cross-split duplicate count: 0
PREPROCESSING STATUS: PASS


,file,physical_rows,comma_only_rows,structured_rows,BENIGN,ATTACK
0,Monday-WorkingHours.pcap_ISCX.csv,529918,0,529918,529918,0
1,Tuesday-WorkingHours.pcap_ISCX.csv,445909,0,445909,432074,13835
2,Wednesday-workingHours.pcap_ISCX.csv,692703,0,692703,440031,252672
3,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,458968,288602,170366,168186,2180
4,Thursday-WorkingHours-Afternoon-Infilteration....,288602,0,288602,288566,36
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,0,191033,189067,1966
6,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,0,225745,97718,128027
7,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,0,286467,127537,158930


## 5. Malformed / Comma-only Rows
Dòng chỉ có delimiter `,` trở thành hàng có toàn bộ raw fields là NaN sau `pd.read_csv()`.
Loader gốc loại chúng ngay sau parsing, trước metadata và missing-value analysis.
WebAttacks có 288.602 dòng như vậy; không gộp vào NaN/Inf thật.
Physical rows trong báo cáo không tính header.

In [4]:
malformed = read_report('malformed_rows_report.json')
display(pd.DataFrame([{k: malformed[k] for k in
                      ['physical_rows', 'comma_only_rows', 'structured_rows']}]))
display(pd.DataFrame(malformed['files']))
assert malformed['physical_rows'] - malformed['comma_only_rows'] == malformed['structured_rows']

,physical_rows,comma_only_rows,structured_rows
0,3119345,288602,2830743


,file,physical_rows,comma_only_rows,structured_rows
0,Monday-WorkingHours.pcap_ISCX.csv,529918,0,529918
1,Tuesday-WorkingHours.pcap_ISCX.csv,445909,0,445909
2,Wednesday-workingHours.pcap_ISCX.csv,692703,0,692703
3,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,458968,288602,170366
4,Thursday-WorkingHours-Afternoon-Infilteration....,288602,0,288602
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,0,191033
6,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,0,225745
7,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,0,286467


## 6. Schema Audit
Chỉ strip tên cột theo loader; pandas phân biệt header trùng bằng `.1`. Không thay schema thêm.

In [5]:
schema = read_report('schema_report.json')
dtype_report = read_report('datatype_report.json')
print('Schema names consistent:', schema['consistent_names'])
display(pd.DataFrame([{'file': item['file'], 'columns': item['columns']}
                      for item in overview]))
display(pd.DataFrame(schema['files']))
display(pd.DataFrame(dtype_report['conversions']))
assert schema['consistent_names']

Schema names consistent: True


,file,columns
0,Monday-WorkingHours.pcap_ISCX.csv,85
1,Tuesday-WorkingHours.pcap_ISCX.csv,85
2,Wednesday-workingHours.pcap_ISCX.csv,85
3,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,85
4,Thursday-WorkingHours-Afternoon-Infilteration....,85
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,85
6,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,85
7,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,85


,file,missing_columns,extra_columns,same_order,type_mismatch
0,Monday-WorkingHours.pcap_ISCX.csv,[],[],True,{}
1,Tuesday-WorkingHours.pcap_ISCX.csv,[],[],True,{'Bwd Packet Length Max': {'reference': 'float...
2,Wednesday-workingHours.pcap_ISCX.csv,[],[],True,{'Bwd Packet Length Max': {'reference': 'float...
3,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,[],[],True,"{'Fwd Avg Bytes/Bulk': {'reference': 'int64', ..."
4,Thursday-WorkingHours-Afternoon-Infilteration....,[],[],True,{'Bwd Packet Length Max': {'reference': 'float...
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,[],[],True,{'Bwd Packet Length Max': {'reference': 'float...
6,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,[],[],True,"{'Idle Max': {'reference': 'float64', 'actual'..."
7,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,[],[],True,{'Bwd Packet Length Max': {'reference': 'float...


""


## 7. Metadata và Label
Giữ Label gốc, binary_label (BENIGN=0, ATTACK=1), nguồn và định danh để truy vết; không đưa các cột này vào model features.

In [6]:
feature_report = read_report('feature_columns.json')
metadata_columns = dtype_report['metadata']
label_columns = dtype_report['labels']
all_numeric_features = feature_report['all_numeric_features']
feature_columns = feature_report['feature_columns']
display(pd.DataFrame({'Group': ['Metadata', 'Labels', 'Raw numeric features', 'Final model features'],
                      'Columns': [metadata_columns, label_columns, all_numeric_features, feature_columns]}))
assert not set(feature_columns).intersection(EXCLUDED + IDENTIFIERS)

,Group,Columns
0,Metadata,"[Flow ID, Source IP, Destination IP, Timestamp..."
1,Labels,"[Label, binary_label]"
2,Raw numeric features,"[Source Port, Destination Port, Protocol, Flow..."
3,Final model features,"[Source Port, Destination Port, Protocol, Flow..."


## 8. Missing / Infinity Analysis
Thống kê này được pipeline tính trên structured rows trước replace Inf/dropna. Không imputation hoặc scaling.

In [7]:
missing_inf = pd.read_csv(REPORT_DIR / 'missing_inf_report.csv')
non_finite_rows = read_report('non_finite_rows.json')
display(missing_inf.loc[(missing_inf['NaN Count'] + missing_inf['Total Inf']) > 0])
display(pd.DataFrame([non_finite_rows]))
print('Actual rows with NaN or Inf:', cleaning['actual_nan_inf_rows'])
print('NaN cells:', int(missing_inf['NaN Count'].sum()))
print('+Inf cells:', int(missing_inf['+Inf'].sum()))
print('-Inf cells:', int(missing_inf['-Inf'].sum()))

,Feature,NaN Count,NaN %,+Inf,-Inf,Total Inf
0,Flow Bytes/s,1358,0.047973,1509,0,1509
17,Flow Packets/s,0,0.000000,2867,0,2867


,rows_with_nan,percentage_rows_with_nan,rows_with_inf,percentage_rows_with_inf
0,1358,0.047973,2867,0.101281


Actual rows with NaN or Inf: 2867
NaN cells: 1358
+Inf cells: 4376
-Inf cells: 0


## 9. Cleaning NaN / Inf
Đã thực hiện qua `clean_data.clean_non_finite_values()` trong phần 4: kiểm tra giới hạn, replace Inf → NaN, dropna. Không chạy cleaning lần hai.

In [8]:
display(pd.DataFrame([cleaning['non_finite']]))
assert cleaning['structured_rows'] - cleaning['rows_removed_nan_inf'] == cleaning['rows_before_duplicate_cleanup']

,rows_before,rows_with_nan_or_inf,percentage_removed,rows_after,max_fraction
0,2830743,2867,0.101281,2827876,0.01


## 10. Label Distribution
Hiển thị dữ liệu cuối sau cleaning/cleanup, không sampling hoặc rebalance.

In [9]:
labels = pd.read_csv(REPORT_DIR / 'label_distribution.csv')
labels_by_day = pd.read_csv(REPORT_DIR / 'label_distribution_by_day.csv')
display(labels)
display(labels_by_day)
print('BENIGN:', int(labels.loc[labels.Label.eq('BENIGN'), 'Count'].sum()))
print('ATTACK:', int(labels.loc[labels.Label.ne('BENIGN'), 'Count'].sum()))

,Label,Count,Percentage
0,BENIGN,2261990,80.470430
1,DoS Hulk,222563,7.917692
2,PortScan,158804,5.649462
3,DDoS,128025,4.554497
4,DoS GoldenEye,10293,0.366174
5,FTP-Patator,7935,0.282288
6,SSH-Patator,5897,0.209786
7,DoS slowloris,5769,0.205233
8,DoS Slowhttptest,5499,0.195627
9,Bot,1956,0.069585


,source_day,Label,Count,Percentage
0,Friday-Bot,BENIGN,187498,98.967559
1,Friday-Bot,Bot,1956,1.032441
2,Friday-DDoS,BENIGN,97323,43.187869
3,Friday-DDoS,DDoS,128025,56.812131
4,Friday-PortScan,BENIGN,126547,44.347838
5,Friday-PortScan,PortScan,158804,55.652162
6,Monday,BENIGN,528082,100.000000
7,Thursday-Infiltration,BENIGN,287226,99.987468
8,Thursday-Infiltration,Infiltration,36,0.012532
9,Thursday-WebAttack,BENIGN,166887,98.710570


BENIGN: 2261990
ATTACK: 548968


## 11. Dataset Split
`split_data.create_dataset_splits()` chạy trước duplicate cleanup:
Monday BENIGN 80% đầu Train / 20% cuối Calibration, không shuffle.
Tuesday + Wednesday + Thursday → Selection; Friday → Test.
Không re-split sau cleanup; vì vậy số record cuối có thể khác ratio ban đầu.

In [10]:
display(statistics_table(cleaning['split_statistics_before_cleanup']))
display(pd.DataFrame([{'Split': name, 'Sources': item['sources']}
                     for name, item in cleaning['split_statistics_before_cleanup'].items()]))

,Split,Rows,BENIGN,ATTACK
0,train,423584,423584,0
1,calibration,105897,105897,0
2,selection,1595677,1327906,267771
3,test,702718,413933,288785


,Split,Sources
0,train,[Monday]
1,calibration,[Monday]
2,selection,"[Thursday-Infiltration, Thursday-WebAttack, Tu..."
3,test,"[Friday-Bot, Friday-DDoS, Friday-PortScan]"


## 12. Duplicate Analysis Before Cleanup
Pipeline đã kiểm tra leakage **trước bất kỳ row dedup nào**, trên toàn bộ raw numeric features
và final model features. Nội bộ từng split: bảng đếm hàng dư theo feature + Label được
ghi lại trước cleanup. Các bảng overlap chứa đủ 6 cặp split.

In [11]:
duplicates = read_report('duplicate_report.json')
leakage = read_report('cross_split_duplicate_report.json')
display(pd.DataFrame([{'Split': name, 'Rows before': item['rows_before'],
                      'Within-split feature + Label duplicates': item['within_split_duplicates_removed']}
                     for name, item in duplicates['cleanup']['splits'].items()]))
display(Markdown('**All raw numeric features — before cleanup**'))
display(pd.DataFrame(leakage['before_all_numeric_features']))
display(Markdown('**Final model features — before cleanup**'))
display(pd.DataFrame(leakage['before']))
display(pd.DataFrame([{'Stage': 'before', **{k: v for k, v in duplicates['before'].items()
                     if k != 'label_conflicts'}}]))

,Split,Rows before,Within-split feature + Label duplicates
0,train,423584,1232
1,calibration,105897,59
2,selection,1595677,12860
3,test,702718,1255


**All raw numeric features — before cleanup**

,Dataset A,Dataset B,Duplicate Feature Rows,matching_rows_a,matching_rows_b,percentage_a,percentage_b
0,train,calibration,108,264,112,0.062325,0.105763
1,train,selection,64,217,619,0.051230,0.038792
2,train,test,26,172,178,0.040606,0.025330
3,calibration,selection,32,32,279,0.030218,0.017485
4,calibration,test,19,20,80,0.018886,0.011384
5,selection,test,1275,4430,2359,0.277625,0.335697


**Final model features — before cleanup**

,Dataset A,Dataset B,Duplicate Feature Rows,matching_rows_a,matching_rows_b,percentage_a,percentage_b
0,train,calibration,108,264,112,0.062325,0.105763
1,train,selection,64,217,619,0.051230,0.038792
2,train,test,26,172,178,0.040606,0.025330
3,calibration,selection,32,32,279,0.030218,0.017485
4,calibration,test,19,20,80,0.018886,0.011384
5,selection,test,1275,4430,2359,0.277625,0.335697


,Stage,full_duplicates,feature_label_duplicates,conflict_groups,conflict_policy
0,before,199,16918,0,Giữ xung đột; chỉ xóa vector feature + Label g...


## 13. Duplicate Cleanup
Pipeline gọi đúng `validation.cleanup_split_duplicates()`.
Ưu tiên **Train > Calibration > Selection > Test**.
Trong từng split giữ hàng đầu theo model features + Label; giữ conflict khác Label.
Giữa split, bỏ vector trùng khỏi split ưu tiên thấp hơn bất kể Label.
Không global-dedup trước kiểm tra leakage, không shuffle hoặc đổi thứ tự giữ hàng.

In [12]:
print(duplicates['cleanup']['within_split_policy'])
print(duplicates['cleanup']['cross_split_policy'])
display(pd.DataFrame([{'Split': name, 'Rows before': item['rows_before'],
                      'Within-split removed': item['within_split_duplicates_removed'],
                      'Overlap removed': item['cross_split_rows_removed'],
                      'Rows after': item['rows_after']}
                     for name, item in duplicates['cleanup']['splits'].items()]))

Giữ hàng đầu theo model features + Label; giữ conflict khác Label trong cùng split.
Giữ split ưu tiên cao, bỏ vector giống nhau khỏi split ưu tiên thấp bất kể Label; không shuffle hoặc phân chia lại Monday.


,Split,Rows before,Within-split removed,Overlap removed,Rows after
0,train,423584,1232,0,422352
1,calibration,105897,59,108,105730
2,selection,1595677,12860,94,1582723
3,test,702718,1255,1310,700153


## 14. Leakage Check After Cleanup
Kiểm tra đầy đủ final model-feature vectors, không gồm label/metadata. Hash chỉ lọc ứng viên, so sánh vector đầy đủ xác nhận overlap.

In [13]:
display(pd.DataFrame(leakage['after']))
assert len(leakage['after']) == 6
assert all(item['Duplicate Feature Rows'] == 0 for item in leakage['after'])

,Dataset A,Dataset B,Duplicate Feature Rows,matching_rows_a,matching_rows_b,percentage_a,percentage_b
0,train,calibration,0,0,0,0.0,0.0
1,train,selection,0,0,0,0.0,0.0
2,train,test,0,0,0,0.0,0.0
3,calibration,selection,0,0,0,0.0,0.0
4,calibration,test,0,0,0,0.0,0.0
5,selection,test,0,0,0,0.0,0.0


## 15. Feature Quality Audit
Kết quả từ `feature_audit.audit_features()` và `select_train_features()` trên Train **trước row cleanup**.
Đây là kết quả đã tính trong phần 4, trình bày ở đây theo bố cục notebook.
Chỉ loại constant và exact duplicate columns; giữ nguyên near-constant/highly-correlated features.
Không có bước lựa chọn feature mới trong notebook.

In [14]:
quality = read_report('feature_quality_report.json')
print('Feature audit source:', quality['selection']['fit_source'])
display(Markdown('**Constant features removed**'))
display(pd.DataFrame({'Feature': quality['selection']['constant_features_removed']}))
display(Markdown('**Exact duplicate columns removed / retained**'))
display(pd.DataFrame(quality['selection']['duplicate_features_removed']))
display(Markdown('**Near-constant — chỉ report**'))
display(pd.DataFrame(quality['near_constant_features']))
display(Markdown('**Highly correlated — chỉ report**'))
display(pd.DataFrame(quality['highly_correlated_pairs']))

Feature audit source: train_before_row_cleanup


**Constant features removed**

,Feature
0,Bwd PSH Flags
1,Fwd URG Flags
2,Bwd URG Flags
3,CWE Flag Count
4,Fwd Avg Bytes/Bulk
5,Fwd Avg Packets/Bulk
6,Fwd Avg Bulk Rate
7,Bwd Avg Bytes/Bulk
8,Bwd Avg Packets/Bulk
9,Bwd Avg Bulk Rate


**Exact duplicate columns removed / retained**

,removed,retained
0,SYN Flag Count,Fwd PSH Flags
1,Fwd Header Length.1,Fwd Header Length
2,Subflow Fwd Packets,Total Fwd Packets
3,Subflow Fwd Bytes,Total Length of Fwd Packets
4,Subflow Bwd Packets,Total Backward Packets
5,Subflow Bwd Bytes,Total Length of Bwd Packets


**Near-constant — chỉ report**

,Feature,Dominant Value,Dominant %
0,Bwd PSH Flags,0.0,100.000000
1,Fwd URG Flags,0.0,100.000000
2,Bwd URG Flags,0.0,100.000000
3,RST Flag Count,0.0,99.983238
4,CWE Flag Count,0.0,100.000000
5,ECE Flag Count,0.0,99.983002
6,Fwd Avg Bytes/Bulk,0.0,100.000000
7,Fwd Avg Packets/Bulk,0.0,100.000000
8,Fwd Avg Bulk Rate,0.0,100.000000
9,Bwd Avg Bytes/Bulk,0.0,100.000000


**Highly correlated — chỉ report**

,Feature A,Feature B,Correlation
0,Flow Duration,Fwd IAT Total,0.997653
1,Flow Duration,Bwd IAT Total,0.972086
2,Total Fwd Packets,Total Backward Packets,0.999319
3,Total Fwd Packets,Total Length of Bwd Packets,0.999376
4,Total Fwd Packets,Subflow Fwd Packets,1.000000
5,Total Fwd Packets,Subflow Bwd Packets,0.999319
6,Total Fwd Packets,Subflow Bwd Bytes,0.999376
7,Total Fwd Packets,act_data_pkt_fwd,0.999500
8,Total Backward Packets,Total Length of Bwd Packets,0.997778
9,Total Backward Packets,Subflow Fwd Packets,0.999319


## 16. Final Feature List
Danh sách giữ nguyên từ pipeline, không scaling.

In [15]:
print('Initial numeric features:', len(all_numeric_features))
print('Removed features:', len(feature_report['removed_features']))
print('Final model features:', len(feature_columns))
display(pd.DataFrame({'Removed feature': feature_report['removed_features']}))
display(pd.DataFrame({'Model feature': feature_columns}))

Initial numeric features: 80
Removed features: 16
Final model features: 64


,Removed feature
0,Bwd PSH Flags
1,Fwd URG Flags
2,Bwd URG Flags
3,CWE Flag Count
4,Fwd Avg Bytes/Bulk
5,Fwd Avg Packets/Bulk
6,Fwd Avg Bulk Rate
7,Bwd Avg Bytes/Bulk
8,Bwd Avg Packets/Bulk
9,Bwd Avg Bulk Rate


,Model feature
0,Source Port
1,Destination Port
2,Protocol
3,Flow Duration
4,Total Fwd Packets
...,...
59,Active Min
60,Idle Mean
61,Idle Std
62,Idle Max


## 17. Final Dataset Statistics

In [16]:
final_statistics = read_report('split_statistics.json')
display(statistics_table(final_statistics))

,Split,Rows,BENIGN,ATTACK
0,train,422352,422352,0
1,calibration,105730,105730,0
2,selection,1582723,1322540,260183
3,test,700153,411368,288785


## 18. Save Outputs
`run()` đã gọi `save_cleaned_dataset()` và `save_splits()` ở phần 4,
lưu bốn Parquet và toàn bộ report/metadata hiện có. Không ghi lại bằng thuật toán khác.
Kiểm tra các file đã tạo; raw giữ nguyên.

In [17]:
output_paths = {name: OUTPUT_DIR / 'splits' / f'{name}.parquet' for name in EXPECTED_COUNTS}
display(pd.DataFrame([{'Split': name, 'File': str(path.relative_to(PROJECT_ROOT)),
                      'Bytes': path.stat().st_size, 'Rows': pq.ParquetFile(path).metadata.num_rows}
                     for name, path in output_paths.items()]))
display(pd.DataFrame({'Report': sorted(p.name for p in REPORT_DIR.iterdir()
                                      if p.suffix in {'.json', '.csv', '.md'})}))

,Split,File,Bytes,Rows
0,train,data\splits\train.parquet,71651754,422352
1,calibration,data\splits\calibration.parquet,17178561,105730
2,selection,data\splits\selection.parquet,211292831,1582723
3,test,data\splits\test.parquet,81588197,700153


,Report
0,basic_statistics.csv
1,cleaning_authorization.json
2,cleaning_report.json
3,constant_features.csv
4,correlation_matrix.csv
5,cross_split_duplicate_after_cleanup.json
6,cross_split_duplicate_before_all_numeric.json
7,cross_split_duplicate_before_cleanup.json
8,cross_split_duplicate_report.json
9,dataset_overview.json


## 19. Final Validation
Đọc lại bốn file Parquet vừa sinh. Assertions giữ số record baseline đã được yêu cầu;
nếu khác, dừng và tìm nguyên nhân, không tự sửa expected counts.
Kiểm tra trực tiếp ngày, nhãn, numeric, hữu hạn, schema và overlap bằng function gốc.

In [18]:
splits = {name: pd.read_parquet(path) for name, path in output_paths.items()}
train_df = splits['train']
calibration_df = splits['calibration']
selection_df = splits['selection']
test_df = splits['test']

assert len(train_df) == 422352
assert len(calibration_df) == 105730
assert len(selection_df) == 1582723
assert len(test_df) == 700153
assert (train_df['binary_label'] == 0).all()
assert (calibration_df['binary_label'] == 0).all()

for name, frame in splits.items():
    assert list(frame.columns) == list(train_df.columns), f'Schema khác: {name}'
    assert [c for c in frame if c not in EXCLUDED + IDENTIFIERS] == feature_columns
    for column in feature_columns:
        assert pd.api.types.is_numeric_dtype(frame[column]), (name, column)
        assert frame[column].notna().all(), (name, column, 'NaN')
        assert np.isfinite(frame[column]).all(), (name, column, 'Inf')

# Gọi validation gốc, không viết lại quy tắc split.
validation.validate_dataset_splits(pd.concat(splits.values(), ignore_index=True), splits, feature_columns)
verified_overlap = validation.check_cross_split_duplicates(splits, feature_columns)
assert all(item['Duplicate Feature Rows'] == 0 for item in verified_overlap)
display(pd.DataFrame(verified_overlap))
assert read_report('validation_summary.json')['status'] == 'PASS'
print('PREPROCESSING REPRODUCTION: PASS')

,Dataset A,Dataset B,Duplicate Feature Rows,matching_rows_a,matching_rows_b,percentage_a,percentage_b
0,train,calibration,0,0,0,0.0,0.0
1,train,selection,0,0,0,0.0,0.0
2,train,test,0,0,0,0.0,0.0
3,calibration,selection,0,0,0,0.0,0.0
4,calibration,test,0,0,0,0.0,0.0
5,selection,test,0,0,0,0.0,0.0


PREPROCESSING REPRODUCTION: PASS


## 20. Conclusion
Notebook tái sử dụng nguyên pipeline `src/`, không đổi cleaning, split, duplicate policy
hoặc feature policy. Run All tái tạo 4 dataset: Train 422.352, Calibration 105.730,
Selection 1.582.723, Test 700.153 record; Train/Calibration chỉ BENIGN và overlap = 0.
Kết luận hợp lệ khi cell validation in **PREPROCESSING REPRODUCTION: PASS**.

Module sử dụng: `config`, `load_data`, `data_audit`, `clean_data`, `feature_audit`,
`split_data`, `validation`, `reporting` và `run_preprocessing`.
Không train model, scaling, PCA, SMOTE, threshold hoặc loại outlier.